### Librerias

In [1]:
# Importar el dataset al notebook
import re
from datetime import date
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import requests
import pandas as pd
import time
import os

# Cargar evidently al entorno
from evidently import Report
from evidently.presets import (
    DataDriftPreset,
    DataSummaryPreset
)

### Variables de entorno

In [2]:
# Fechas de inicio y fin del dataset de referencia
inicio_r = date(2023, 1, 1)
fin_r = date(2026, 9, 30)

# Fechas de inicio y fin del dataser de producción
inicio_p = date(2026, 10, 1)
fin_p = date(2026, 10, 31)

# Pagina web de donde se obtendrán los datos
pagina = "https://mibici.net/es/datos-abiertos/"

# Carga de datos al notebook

In [3]:
sesion = requests.Session()
sesion.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-MX,es;q=0.9,en;q=0.8",
})

respuesta = sesion.get(pagina, timeout=30)
respuesta.raise_for_status()

In [4]:
def descargar_archivos(carpeta_i, inicio, fin):
    soup = BeautifulSoup(respuesta.text, "html.parser")
    if not os.path.exists("data"):
        os.makedirs("data")
    carpeta = Path(f"data/{carpeta_i}")
    carpeta.mkdir(exist_ok=True)
    for a in soup.select('a[href$=".csv"]'):
        url = urljoin(pagina, a["href"])
        nombre = url.rsplit("/", 1)[-1]

        coincidencia = re.search(r"datos_abiertos_(\d{4})_(\d{2})\.csv$", nombre)
        if not coincidencia:
            print("Omitido:", nombre)
            continue

        anio, mes = map(int, coincidencia.groups())
        mes_inicio = date(anio, mes, 1)
        siguiente_mes = date(anio + (mes == 12), mes % 12 + 1, 1)
        mes_fin = date.fromordinal(siguiente_mes.toordinal() - 1)
        
        destino = carpeta / nombre
        if mes_inicio <= fin and mes_fin >= inicio:    
            for intento in range(5):
                try:
                    with sesion.get(
                        url,
                        timeout=(20, 180),
                        stream=True
                    ) as r:
                        r.raise_for_status()

                        with open(destino, "wb") as archivo:
                            for bloque in r.iter_content(chunk_size=1024 * 1024):
                                if bloque:
                                    archivo.write(bloque)

                    print("Descargado:", nombre)
                    break

                except requests.exceptions.RequestException:
                    destino.unlink(missing_ok=True)

                    if intento == 4:
                        raise

                    print(f"Descarga interrumpida; reintentando ({intento + 1}/4)…")
                    time.sleep(2 ** intento)

In [5]:
def cargar_datos(carpeta_i):
    carpeta = Path(f"data/{carpeta_i}")
    archivos = sorted(carpeta.glob("datos_abiertos_*.csv"))
    dataframes = []

    for archivo in archivos:
        try:
            tabla = pd.read_csv(archivo, encoding="latin-1")
            print(archivo.name, "latin-1")
        except UnicodeDecodeError:
            try:
                tabla = pd.read_csv(archivo, encoding="UTF-8")
                print(archivo.name, "UTF-8")
            except UnicodeDecodeError:
                print(archivo.name, "necesita otra codificación")
                continue

        dataframes.append(tabla)

    df = pd.concat(dataframes, ignore_index=True)

    return df

In [6]:
# Descargar archivos de referencia
descargar_archivos("referencia", inicio_r, fin_r)

# Descargar archivos de producción
descargar_archivos("produccion", inicio_p, fin_p)


Omitido: nomenclatura_2026_10.csv
Omitido: datos_abiertos_2016_01-1.csv
Omitido: datos_abiertos_2017_04-1.csv
Descargado: datos_abiertos_2023_01.csv
Descargado: datos_abiertos_2023_02.csv
Descargado: datos_abiertos_2023_03.csv
Descargado: datos_abiertos_2023_04.csv
Descargado: datos_abiertos_2023_05.csv
Descargado: datos_abiertos_2023_06.csv
Descargado: datos_abiertos_2023_07.csv
Descargado: datos_abiertos_2023_08.csv
Descargado: datos_abiertos_2023_09.csv
Descargado: datos_abiertos_2023_10.csv
Descargado: datos_abiertos_2023_11.csv
Descargado: datos_abiertos_2023_12.csv
Descargado: datos_abiertos_2024_01.csv
Descargado: datos_abiertos_2024_02.csv
Descargado: datos_abiertos_2024_03.csv
Descargado: datos_abiertos_2024_04.csv
Descargado: datos_abiertos_2024_05.csv
Descargado: datos_abiertos_2024_06.csv
Descargado: datos_abiertos_2024_07.csv
Descargado: datos_abiertos_2024_08.csv
Descargado: datos_abiertos_2024_09.csv
Descargado: datos_abiertos_2024_10.csv
Descargado: datos_abiertos_2024_

In [7]:
df_referencia = cargar_datos("referencia")
df_produccion = cargar_datos("produccion")

datos_abiertos_2023_01.csv latin-1
datos_abiertos_2023_02.csv latin-1
datos_abiertos_2023_03.csv latin-1
datos_abiertos_2023_04.csv latin-1
datos_abiertos_2023_05.csv latin-1
datos_abiertos_2023_06.csv latin-1
datos_abiertos_2023_07.csv latin-1
datos_abiertos_2023_08.csv latin-1
datos_abiertos_2023_09.csv latin-1
datos_abiertos_2023_10.csv latin-1
datos_abiertos_2023_11.csv latin-1
datos_abiertos_2023_12.csv latin-1
datos_abiertos_2024_01.csv latin-1
datos_abiertos_2024_02.csv latin-1
datos_abiertos_2024_03.csv latin-1
datos_abiertos_2024_04.csv latin-1
datos_abiertos_2024_05.csv latin-1
datos_abiertos_2024_06.csv latin-1
datos_abiertos_2024_07.csv latin-1
datos_abiertos_2024_08.csv latin-1
datos_abiertos_2024_09.csv latin-1
datos_abiertos_2024_10.csv latin-1
datos_abiertos_2024_11.csv latin-1
datos_abiertos_2024_12.csv latin-1
datos_abiertos_2025_01.csv latin-1
datos_abiertos_2025_02.csv latin-1
datos_abiertos_2025_03.csv latin-1
datos_abiertos_2025_04.csv latin-1
datos_abiertos_2025_

# Generación de reporte con Evidently

In [8]:
# Asegurar que ambos DataFrames tengan las mismas columnas
common_columns = df_referencia.columns.intersection(
    df_produccion.columns
)

reference_data = df_referencia[common_columns]
current_data = df_produccion[common_columns]

In [9]:
# Definir el reporte
report = Report(
    metrics=[
        DataDriftPreset(),
        DataSummaryPreset()
    ],
    include_tests=True
)

In [11]:
# Ejecutar la comparación
snapshot = report.run(
    current_data= current_data,
    reference_data=reference_data
)

In [ ]:
# Exportar el reporte HTML
if not os.path.exists("reports"):
        os.makedirs("reports")
snapshot.save_html("reports/drift_report_octubre2026.html")

print("Reporte generado correctamente.")

Reporte generado correctamente.
